# An Efficient Arthritis Diagnostic Model using CoAtNet by Integrating Convolutional and Attention Mechanisms for Enhanced Precision

Official code for the paper by A. Shehzad, F. Akbar, T. Kehkashan, M. Mohsin, M. Zaman and A. R. Shoukat, published at the 2024 5th International Conference on Innovative Computing (ICIC), IEEE. DOI: [10.1109/ICIC63915.2024.11116318](https://doi.org/10.1109/ICIC63915.2024.11116318)

**Pipeline (follows Section III, Methodology)**
1. **Data collection (III-A):** Kaggle Knee Osteoarthritis Dataset, knee X-rays labelled by Kellgren–Lawrence (KL) grade 0–4.
2. **Data distribution (III-A-2):** 8,000 images split 60/20/20, giving 4,800 train, 1,600 validation and 1,600 test.
3. **Preprocessing (III-B):** resize to 224×224×3, normalization, augmentation (rotation, horizontal/vertical flip, zoom, shift), one-hot label encoding.
4. **Model (III-C, Fig. 1):** CoAtNet: Conv stem (Conv2D + BatchNorm + GELU) → MBConv + SE → Transformer (relative-position MHA) ×2 → global average pooling → fully connected → softmax.
5. **Training (III-D):** AdamW, learning rate 0.001, batch size 32, 50 epochs, cross-entropy loss.
6. **Results (IV):** accuracy and loss curves (Figs. 2–3), confusion matrix (Fig. 4), per-class ROC (Fig. 5), Table I comparison.

**Runs on:** Kaggle or Google Colab with a GPU (T4 or better). On Kaggle, attach the dataset `shashwatwork/knee-osteoarthritis-dataset-with-severity`; on Colab it is downloaded automatically through `kagglehub`.

> Implementation note: CoAtNet is loaded from PyTorch `timm` (`coatnet_0_rw_224`, ImageNet-pretrained). Its stage layout matches Fig. 1 of the paper: C-C-T-T, with MBConv + SE blocks followed by relative-position transformer blocks.

## 1. Setup

In [ ]:
!pip install -q timm kagglehub

In [ ]:
import os, re, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report, roc_curve, auc, roc_auc_score)

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| timm", timm.__version__)

## 2. Configuration
Training parameters follow Section III-D of the paper.

In [ ]:
SEED = 42

# Dataset location. Leave as None to auto-detect (/kaggle/input) or auto-download with kagglehub.
DATA_DIR = None
KAGGLE_DATASET = "shashwatwork/knee-osteoarthritis-dataset-with-severity"
EXCLUDE_DIRS = {"auto_test"}     # auto_test duplicates the test images and is skipped

KL_NAMES = ["Grade 0: No OA", "Grade 1: Probable OA", "Grade 2: Minimal OA",
            "Grade 3: Moderate OA", "Grade 4: Severe OA"]
SHORT_NAMES = ["KL0", "KL1", "KL2", "KL3", "KL4"]

# Data distribution (paper III-A-2): 8000 images -> 60% train / 20% val / 20% test
TOTAL_IMAGES = 8000        # set to None to use every available image
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.60, 0.20, 0.20

# Preprocessing (paper III-B)
IMG_SIZE = 224

# Model and training (paper III-C, III-D)
MODEL_NAME = "coatnet_0_rw_224.sw_in1k"   # CoAtNet-0, ImageNet-1k pretrained
PRETRAINED = True
LR = 1e-3                  # paper: 0.001 (lower to 1e-4 if training is unstable)
BATCH_SIZE = 32            # paper: 32
EPOCHS = 50                # paper: 50
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 3          # linear warm-up, then cosine decay
PATIENCE = 10              # early stopping on validation loss
NUM_WORKERS = 2

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## 3. Data collection (Section III-A)
Collect every image in folders named `0`–`4` (the KL grade). The dataset's own train/val/test folders are pooled, duplicates removed, and the data re-split 60/20/20 as described in the paper.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

if DATA_DIR is None:
    if os.path.isdir("/kaggle/input"):
        DATA_DIR = "/kaggle/input"
    else:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download(KAGGLE_DATASET)

records = []
for dirpath, dirnames, filenames in os.walk(DATA_DIR):
    dirnames[:] = [d for d in dirnames if d.lower() not in EXCLUDE_DIRS]
    grade = os.path.basename(dirpath)
    if not re.fullmatch(r"[0-4]", grade):
        continue
    for f in filenames:
        if os.path.splitext(f)[1].lower() in IMG_EXT:
            records.append((os.path.join(dirpath, f), int(grade), f))

all_df = pd.DataFrame(records, columns=["path", "label", "fname"])
assert len(all_df), f"No KL-grade folders (0-4) found under {DATA_DIR}"
before = len(all_df)
all_df = all_df.drop_duplicates(subset=["fname", "label"]).reset_index(drop=True)
print(f"Found {before} images, {len(all_df)} after removing duplicates")

NUM_CLASSES = all_df["label"].nunique()
assert NUM_CLASSES == len(KL_NAMES), f"Expected 5 KL grades, found {NUM_CLASSES}"
print(all_df["label"].value_counts().sort_index().rename(dict(enumerate(KL_NAMES))))

with Image.open(all_df.path.iloc[0]) as im:
    print("Example image size / mode:", im.size, im.mode)

### Data distribution (Section III-A-2)
Stratified 60 / 20 / 20 split: 4,800 train, 1,600 validation, 1,600 test.

In [ ]:
df = all_df
if TOTAL_IMAGES is not None and len(df) > TOTAL_IMAGES:
    df, _ = train_test_split(df, train_size=TOTAL_IMAGES, stratify=df["label"], random_state=SEED)
df = df.reset_index(drop=True)

train_df, temp_df = train_test_split(df, train_size=TRAIN_FRAC, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                                   stratify=temp_df["label"], random_state=SEED)

split_table = pd.DataFrame({
    name: d["label"].value_counts().sort_index() for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]
}).rename(index=dict(enumerate(SHORT_NAMES)))
split_table.loc["Total"] = split_table.sum()
split_table

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for c in range(NUM_CLASSES):
    samples = df[df.label == c].sample(2, random_state=SEED)
    for r, p in enumerate(samples.path):
        axes[r, c].imshow(Image.open(p).convert("L"), cmap="gray"); axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(KL_NAMES[c], fontsize=10)
plt.suptitle("Sample knee X-rays per KL grade"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sample_images.png", dpi=300, bbox_inches="tight"); plt.show()

## 4. Model architecture: CoAtNet (Section III-C, Fig. 1)

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=NUM_CLASSES).to(DEVICE)
data_cfg = timm.data.resolve_model_data_config(model)
MEAN, STD = data_cfg["mean"], data_cfg["std"]

print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("Stem:", model.stem.__class__.__name__)
for i, stage in enumerate(model.stages):
    blk = stage.blocks[0]
    print(f"Stage {i+1}: {blk.__class__.__name__:18s} x{len(stage.blocks)}")
print("Head:", model.head)

## 5. Preprocessing (Section III-B)
1. **Image resizing:** 224×224, with grayscale replicated to 3 channels (224×224×3 input).
2. **Normalization:** scale to [0, 1], then standardize with the pretrained mean/std.
3. **Data augmentation (train only):** random rotation, horizontal and vertical flip, zoom and shift.
4. **Label encoding:** KL grades are one-hot encoded inside the loss (`F.one_hot`).

In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),                                        # rotation
    transforms.RandomHorizontalFlip(),                                    # horizontal flip
    transforms.RandomVerticalFlip(),                                      # vertical flip
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # shift + zoom
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class KneeXrayDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")
        return self.transform(img), self.labels[i]

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(KneeXrayDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=USE_AMP, generator=g, drop_last=True)
val_loader = DataLoader(KneeXrayDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=USE_AMP)
test_loader = DataLoader(KneeXrayDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=USE_AMP)

xb, yb = next(iter(train_loader))
print("Batch:", tuple(xb.shape), "| one-hot labels:", tuple(F.one_hot(yb, NUM_CLASSES).shape))

In [ ]:
# Augmented training samples
inv = transforms.Normalize([-m / s for m, s in zip(MEAN, STD)], [1 / s for s in STD])
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, x, y in zip(axes, xb[:6], yb[:6]):
    ax.imshow(inv(x).permute(1, 2, 0).clamp(0, 1).numpy()); ax.set_title(SHORT_NAMES[y]); ax.axis("off")
plt.suptitle("Augmented training images"); plt.tight_layout(); plt.show()

## 6. Training (Section III-D)
AdamW (lr = 0.001), batch size 32, 50 epochs, cross-entropy loss on one-hot labels. The best checkpoint by validation loss is kept.

In [ ]:
criterion = nn.CrossEntropyLoss()   # accepts one-hot (probability) targets
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(enabled=USE_AMP)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False, desc="train" if train else "val"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            y_onehot = F.one_hot(y, NUM_CLASSES).float()
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y_onehot)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

In [ ]:
BEST_PATH = f"{OUT_DIR}/coatnet_knee_oa_best.pth"
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss, wait = float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
        history[k].append(v)

    improved = va_loss < best_val_loss
    if improved:
        best_val_loss, wait = va_loss, 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        wait += 1
    print(f"Epoch {epoch:02d}/{EPOCHS} | lr {optimizer.param_groups[0]['lr']:.2e} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}"
          + ("  *best*" if improved else ""))
    if wait >= PATIENCE:
        print(f"Early stopping (no val-loss improvement for {PATIENCE} epochs).")
        break

model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
print("Loaded best model from", BEST_PATH)

In [ ]:
# Fig. 2: training and validation accuracy | Fig. 3: training and validation loss
ep = range(1, len(history["train_loss"]) + 1)
for key, title, fname in [("acc", "Training and Validation Accuracy", "fig2_accuracy.png"),
                          ("loss", "Training and Validation Loss", "fig3_loss.png")]:
    plt.figure(figsize=(7, 4.5))
    plt.plot(ep, history[f"train_{key}"], label=f"Training {key.replace('acc', 'Accuracy').replace('loss', 'Loss')}")
    plt.plot(ep, history[f"val_{key}"], label=f"Validation {key.replace('acc', 'Accuracy').replace('loss', 'Loss')}")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy" if key == "acc" else "Loss"); plt.title(title)
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fname}", dpi=300, bbox_inches="tight"); plt.show()

## 7. Results and discussion (Section IV)
Test-set evaluation over the five KL grades. Precision, recall and F1 are macro-averaged; weighted averages and per-class scores are also reported.

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(DEVICE))
        ps.append(logits.float().softmax(1).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

y_true, y_prob = predict(test_loader)
y_pred = y_prob.argmax(1)
y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision (macro)": p_mac, "Recall (macro)": r_mac, "F1 (macro)": f_mac,
    "Precision (weighted)": p_w, "Recall (weighted)": r_w, "F1 (weighted)": f_w,
    "AUC (macro OvR)": roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr"),
    "Top-2 Accuracy": float(np.mean([t in np.argsort(p)[-2:] for t, p in zip(y_true, y_prob)])),
}
print(f"Test images: {len(y_true)}\n")
for k, v in metrics.items():
    print(f"{k:22s}: {v:.4f}")
print("\n" + classification_report(y_true, y_pred, target_names=KL_NAMES, digits=4, zero_division=0))

In [ ]:
# Fig. 4: confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES, ax=ax[0])
ax[0].set_title("Confusion Matrix (counts)"); ax[0].set_xlabel("Predicted Labels"); ax[0].set_ylabel("True Labels")
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES, ax=ax[1], vmin=0, vmax=1)
ax[1].set_title("Confusion Matrix (row-normalized)"); ax[1].set_xlabel("Predicted Labels"); ax[1].set_ylabel("True Labels")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_confusion_matrix.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# Fig. 5: ROC curve for each class (one-vs-rest) + micro / macro average
plt.figure(figsize=(7.5, 6))
fpr_grid, tprs = np.linspace(0, 1, 500), []
for c in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, c], y_prob[:, c])
    tprs.append(np.interp(fpr_grid, fpr, tpr))
    plt.plot(fpr, tpr, lw=1.6, label=f"ROC curve of class {c} (area = {auc(fpr, tpr):.2f})")
fpr_mi, tpr_mi, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
plt.plot(fpr_mi, tpr_mi, "k:", lw=2.5, label=f"Micro-average (area = {auc(fpr_mi, tpr_mi):.2f})")
plt.plot(fpr_grid, np.mean(tprs, 0), "k--", lw=2, label=f"Macro-average (area = {auc(fpr_grid, np.mean(tprs, 0)):.2f})")
plt.plot([0, 1], [0, 1], color="grey", lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic for Each Class"); plt.legend(loc="lower right", fontsize=9); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_roc_curve.png", dpi=300, bbox_inches="tight"); plt.show()

### Table I: comparison with state-of-the-art studies
Comparison-model values are taken from Table I of the paper; the CoAtNet row comes from this run.

In [ ]:
table1 = pd.DataFrame([
    ["CoAtNet (this run)", round(metrics["Accuracy"] * 100, 1), round(metrics["F1 (macro)"], 2),
     round(metrics["Precision (macro)"], 2), round(metrics["Recall (macro)"], 2)],
    ["DenseNet-121", 77.6, 0.78, 0.79, 0.77],
    ["EfficientNet", 84.3, 0.83, 0.84, 0.82],
    ["ViT",          85.0, 0.86, 0.85, 0.85],
    ["ResNet-50",    74.4, 0.75, 0.77, 0.76],
], columns=["Studies", "Acc", "F1 score", "Precision", "Recall"])
table1.to_csv(f"{OUT_DIR}/table1_comparison.csv", index=False)
table1

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(table1)); w = 0.2
for i, col in enumerate(["F1 score", "Precision", "Recall"]):
    ax1.bar(x + (i - 1) * w, table1[col], w, label=col)
ax1.set_ylabel("Score"); ax1.set_ylim(0, 1.05); ax1.set_xticks(x); ax1.set_xticklabels(table1["Studies"])
ax2 = ax1.twinx(); ax2.plot(x, table1["Acc"], "ko-", label="Accuracy (%)"); ax2.set_ylim(0, 105); ax2.set_ylabel("Accuracy (%)")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower right", ncol=4); plt.title("Comparison with state-of-the-art models")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/table1_comparison_chart.png", dpi=300, bbox_inches="tight"); plt.show()

## 8. Interpretability: Grad-CAM on the final CoAtNet stage
Shows which knee regions (joint space, osteophytes) drive the predicted KL grade. The paper discusses interpretability but does not specify a method; this section is an addition.

In [ ]:
def grad_cam(model, img_tensor, class_index=None, target_layer=None):
    model.eval()
    target_layer = target_layer or model.stages[-1]
    store = {}
    def hook(module, inputs, output):
        store["act"] = output
        output.register_hook(lambda g: store.__setitem__("grad", g))
    h = target_layer.register_forward_hook(hook)
    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(img_tensor.unsqueeze(0).to(DEVICE))
            probs = logits.softmax(1)[0].detach().cpu().numpy()
            class_index = int(probs.argmax()) if class_index is None else class_index
            logits[0, class_index].backward()
    finally:
        h.remove()
    act, grad = store["act"][0].detach(), store["grad"][0].detach()      # (C, H, W)
    if act.dim() == 3 and act.shape[-1] == grad.shape[-1] and act.shape[0] < act.shape[-1]:
        act, grad = act.permute(2, 0, 1), grad.permute(2, 0, 1)          # handle channels-last outputs
    weights = grad.mean(dim=(1, 2), keepdim=True)
    cam = F.relu((weights * act).sum(0)).cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, class_index, probs

def show_gradcam(frame, n=5, fname="gradcam.png", title="Grad-CAM (CoAtNet, final stage)"):
    picks = frame.groupby("label", group_keys=False).apply(lambda d: d.sample(1, random_state=SEED)).head(n)
    fig, axes = plt.subplots(2, len(picks), figsize=(3.3 * len(picks), 6.8))
    axes = np.array(axes).reshape(2, -1)
    for j, (path, label) in enumerate(zip(picks.path, picks.label)):
        img = np.asarray(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE)))
        cam, pred, probs = grad_cam(model, eval_tf(Image.fromarray(img)))
        cam_up = np.asarray(Image.fromarray(np.uint8(cam * 255)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
        axes[0, j].imshow(img, cmap="gray"); axes[0, j].set_title(f"True: {SHORT_NAMES[label]}")
        axes[1, j].imshow(img, cmap="gray"); axes[1, j].imshow(cam_up, cmap="jet", alpha=0.45)
        axes[1, j].set_title(f"Pred: {SHORT_NAMES[pred]} ({probs[pred]*100:.1f}%)", color="green" if pred == label else "red")
        for a in axes[:, j]:
            a.axis("off")
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fname}", dpi=300, bbox_inches="tight"); plt.show()

show_gradcam(test_df, n=NUM_CLASSES)

## 9. Save results

In [ ]:
test_eval = test_df.reset_index(drop=True).assign(pred=y_pred, confidence=y_prob.max(1))
results = {
    "model": MODEL_NAME, "epochs_trained": len(history["train_loss"]),
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "metrics": {k: round(float(v), 4) for k, v in metrics.items()},
    "confusion_matrix": cm.tolist(),
    "history": history,
}
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
test_eval.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
print(json.dumps(results["metrics"], indent=2))
print("Saved to:", sorted(os.listdir(OUT_DIR)))